# Verifikasi WER — Checkpoint-2000 (Whisper Fine-Tuned)

Notebook ini **berdiri sendiri**, tidak bergantung ke Colab/Drive. Tujuannya: membuktikan checkpoint-2000 yang sudah ada di folder proyek NDTW (dan sudah dipakai untuk web & mobile) benar-benar menghasilkan WER ~12,79% pada test set, sebagai bukti independen — terlepas dari kondisi checkpoint di Google Drive/Colab.

**Sebelum menjalankan, edit sel CONFIG di bawah** sesuai kondisi laptop kamu:
- Path checkpoint-2000
- Path file test set (CSV) + nama kolom audio & teks referensi
- Path folder audio (jika kolom audio berisi nama file relatif, bukan path lengkap)

**Cara pakai yang disarankan:**
1. Jalankan dulu dengan `SUBSET_N = 20` (cepat, cek pipeline jalan benar & angka masuk akal).
2. Kalau lolos, ubah `SUBSET_N = None` untuk jalankan ke seluruh 3.694 sampel test set (bisa makan waktu — lihat estimasi di sel inferensi).


In [1]:
# Sel 1 — Install dependency (jalankan sekali saja, boleh di-skip kalau sudah terpasang)
# !pip install torch transformers jiwer librosa soundfile pandas tqdm


In [2]:
# Sel 2 — CONFIG (WAJIB DIEDIT SESUAI KONDISI LAPTOP KAMU)

CHECKPOINT_PATH = "../model/whisper_finetuned"  # <-- sesuai screenshot: ndtw_project/model/whisper_finetuned

TEST_CSV_PATH = "../data/common_voice_id/test.tsv"  # <-- test split resmi Common Voice (3.694 sampel)
TSV_SEP = "\t"                                    # Common Voice pakai TAB, bukan koma
AUDIO_COL = "path"                                  # <-- kolom nama file audio di Common Voice mentah (BUKAN "audio")
TEXT_COL = "sentence"                               # kolom teks referensi (ground truth)
AUDIO_BASE_DIR = "../data/common_voice_id/clips"    # folder audio .mp3 Common Voice

ORIGINAL_SR = 32000   # sample rate asli file Common Voice hasil download manual kamu
TARGET_SR = 16000     # Whisper WAJIB input 16kHz — resample dilakukan otomatis di bawah

SUBSET_N = 20          # <-- mulai dari angka kecil dulu untuk sanity check; set None untuk full 3.694 sampel

DEVICE_PREFERENCE = "cuda"  # otomatis fallback ke "cpu" kalau GPU tidak ada
USE_FP16 = True        # <-- aktifkan biar model lebih hemat VRAM (penting untuk MX130 2GB); otomatis diabaikan kalau device=cpu


In [3]:
# Sel 2b — Diagnostik GPU (jalankan ini DULU untuk cek apakah MX130 terdeteksi)
import torch
print("PyTorch version   :", torch.__version__)
print("CUDA tersedia?    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nama GPU          :", torch.cuda.get_device_name(0))
    print(f"Total VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print()
    print("GPU TIDAK terdeteksi. Kemungkinan penyebab:")
    print("1. PyTorch yang terpasang adalah versi CPU-only (paling sering).")
    print("   Cek dulu driver NVIDIA: jalankan  di terminal (lihat versi CUDA driver di pojok kanan atas).")
    print("   Lalu install ulang PyTorch versi CUDA, contoh untuk CUDA 11.8 (kompatibel luas termasuk GPU lama seperti MX130):")
    print("   pip uninstall torch")
    print("   pip install torch --index-url https://download.pytorch.org/whl/cu118")
    print("2. Driver NVIDIA belum terpasang/kadaluarsa -> update lewat NVIDIA GeForce Experience atau situs resmi NVIDIA.")


PyTorch version   : 2.13.0+cpu
CUDA tersedia?    : False

GPU TIDAK terdeteksi. Kemungkinan penyebab:
1. PyTorch yang terpasang adalah versi CPU-only (paling sering).
   Cek dulu driver NVIDIA: jalankan  di terminal (lihat versi CUDA driver di pojok kanan atas).
   Lalu install ulang PyTorch versi CUDA, contoh untuk CUDA 11.8 (kompatibel luas termasuk GPU lama seperti MX130):
   pip uninstall torch
   pip install torch --index-url https://download.pytorch.org/whl/cu118
2. Driver NVIDIA belum terpasang/kadaluarsa -> update lewat NVIDIA GeForce Experience atau situs resmi NVIDIA.


In [4]:
# Sel 3 — Load model & processor dari checkpoint LOKAL (tidak menyentuh internet/Colab sama sekali)
import os
import torch
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperTokenizer,
    WhisperFeatureExtractor,
)

device = DEVICE_PREFERENCE if torch.cuda.is_available() else "cpu"
print(f"Menggunakan device: {device}")
if device == "cpu":
    print("PERINGATAN: berjalan di CPU akan jauh lebih lambat. Untuk full 3.694 sampel, pertimbangkan pakai GPU kalau ada.")

BASE_MODEL_FOR_TOKENIZER = "openai/whisper-small"  # dipakai HANYA untuk tokenizer, kalau checkpoint tidak menyertakannya

# Deteksi otomatis: apakah folder checkpoint punya file tokenizer sendiri?
has_tokenizer_files = any(
    os.path.exists(os.path.join(CHECKPOINT_PATH, f))
    for f in ["tokenizer.json", "vocab.json", "tokenizer_config.json"]
)

if has_tokenizer_files:
    print("Tokenizer ditemukan di dalam folder checkpoint, memuat langsung dari sana.")
    processor = WhisperProcessor.from_pretrained(CHECKPOINT_PATH)
else:
    print(f"Tokenizer TIDAK ditemukan di {CHECKPOINT_PATH} (checkpoint hanya menyimpan model + feature extractor).")
    print(f"Mengambil tokenizer dari base model '{BASE_MODEL_FOR_TOKENIZER}' (aman, fine-tuning tidak mengubah vocab).")
    print("Catatan: butuh koneksi internet SEKALI SAJA kalau base model ini belum pernah ter-cache di laptop kamu.")
    tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL_FOR_TOKENIZER, language="indonesian", task="transcribe")
    feature_extractor = WhisperFeatureExtractor.from_pretrained(CHECKPOINT_PATH)  # ini AMAN diambil dari checkpoint (preprocessor_config.json ada)
    processor = WhisperProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

model_dtype = torch.float16 if (device == "cuda" and USE_FP16) else torch.float32
print(f"Memuat model dengan dtype: {model_dtype}")
model = WhisperForConditionalGeneration.from_pretrained(CHECKPOINT_PATH, torch_dtype=model_dtype).to(device)
model.eval()
print("Checkpoint model berhasil dimuat dari:", CHECKPOINT_PATH)


Menggunakan device: cpu
PERINGATAN: berjalan di CPU akan jauh lebih lambat. Untuk full 3.694 sampel, pertimbangkan pakai GPU kalau ada.
Tokenizer TIDAK ditemukan di ../model/whisper_finetuned (checkpoint hanya menyimpan model + feature extractor).
Mengambil tokenizer dari base model 'openai/whisper-small' (aman, fine-tuning tidak mengubah vocab).
Catatan: butuh koneksi internet SEKALI SAJA kalau base model ini belum pernah ter-cache di laptop kamu.
Memuat model dengan dtype: torch.float32


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Checkpoint model berhasil dimuat dari: ../model/whisper_finetuned


In [5]:
# Sel 4 — Load test set
import pandas as pd
import os

df = pd.read_csv(TEST_CSV_PATH, sep=TSV_SEP)
print(f"Total baris di test set: {len(df)}")
print(f"Kolom tersedia: {list(df.columns)}")
assert AUDIO_COL in df.columns, f"Kolom '{AUDIO_COL}' tidak ditemukan. Kolom tersedia: {list(df.columns)}"
assert TEXT_COL in df.columns, f"Kolom '{TEXT_COL}' tidak ditemukan. Kolom tersedia: {list(df.columns)}"

if SUBSET_N is not None:
    df = df.sample(n=min(SUBSET_N, len(df)), random_state=42).reset_index(drop=True)
    print(f"Mode subset aktif: mengambil {len(df)} sampel acak untuk sanity check cepat.")

df.head()


Total baris di test set: 3694
Kolom tersedia: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']
Mode subset aktif: mengambil 20 sampel acak untuk sanity check cepat.


,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,f84257d18d4a39717abf1c83c30b250d584b4416e28aae...,common_voice_id_23583180.mp3,2ccec510bd667bb0ce97c82d4974eea95e4ea9a1b716da...,Kamu di mana?,NaN,2,0,twenties,female_feminine,NaN,NaN,id,NaN
1,1c2e3ced0fbd6a51b39e999cddf31f399289f2a43d9371...,common_voice_id_20953904.mp3,fcb3e6fd700ec5769c68e73548f9075ebcb11d63dd9663...,Tom adalah orang yang menanam mawar di kebun k...,NaN,2,0,teens,male_masculine,NaN,NaN,id,NaN
2,90fdca2d9a7ecd353580a0e7ea467924b4e59ef225a19d...,common_voice_id_24023126.mp3,831ac94327bb422d1f1238cf58d6ca83e32511f76be91d...,Hidup cuma sekali,NaN,2,0,NaN,NaN,NaN,NaN,id,NaN
3,87dd21de1293ab5d7dbe9ebc27722c607b0525f2bcce8d...,common_voice_id_27652237.mp3,429d64c0322b45c026620a578fd3015e15cf974312c4f1...,Dari mana kamu mendapatkan sepatu aneh itu?,NaN,2,0,twenties,male_masculine,NaN,NaN,id,NaN
4,306ce210e3732a4100f8e532e63a052ec048dfbcde726e...,common_voice_id_35276909.mp3,3079f085bdffb3b4461695f81989af280d5f13e6761bc6...,"Nama ""Kleopatra"" menjadi persamaan kata dengan...",NaN,2,0,twenties,male_masculine,KBBI,NaN,id,NaN


In [6]:
# Sel 5 — Fungsi load audio + resample ke 16kHz (WAJIB, karena file asli 32kHz)
import librosa
import numpy as np

def load_audio_16k(row):
    audio_field = str(row[AUDIO_COL])
    path = os.path.join(AUDIO_BASE_DIR, audio_field) if AUDIO_BASE_DIR else audio_field
    speech, sr = librosa.load(path, sr=TARGET_SR)  # librosa.load dengan sr=16000 otomatis resample dari 32kHz
    return speech

# Uji cepat pada 1 sampel untuk pastikan path & resample benar sebelum loop penuh
_test_audio = load_audio_16k(df.iloc[0])
print("Contoh audio berhasil dimuat, panjang (detik):", len(_test_audio) / TARGET_SR)


Contoh audio berhasil dimuat, panjang (detik): 3.408


In [7]:
# Sel 6 — Inference loop (transkripsi seluruh/sebagian test set)
from tqdm import tqdm
import time

predictions = []
references = []
failed_rows = []

start = time.time()
for i, row in tqdm(df.iterrows(), total=len(df), desc="Transkripsi"):
    try:
        speech = load_audio_16k(row)
        inputs = processor(speech, sampling_rate=TARGET_SR, return_tensors="pt").to(device, model_dtype)
        with torch.no_grad():
            predicted_ids = model.generate(inputs["input_features"])
        pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        predictions.append(pred_text)
        references.append(str(row[TEXT_COL]))
    except Exception as e:
        failed_rows.append((i, str(e)))

elapsed = time.time() - start
print(f"Selesai {len(predictions)}/{len(df)} sampel dalam {elapsed:.1f} detik (~{elapsed/max(len(df),1):.2f} detik/sampel)")
if failed_rows:
    print(f"PERINGATAN: {len(failed_rows)} sampel gagal diproses (cek path audio). Contoh: {failed_rows[:3]}")


Transkripsi:   0%|          | 0/20 [00:00<?, ?it/s][transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogi

Selesai 20/20 sampel dalam 133.5 detik (~6.67 detik/sampel)


In [8]:
# Sel 7 — Normalisasi teks (untuk WER 'normalized', sesuai istilah yang dulu dipakai)
import re
import string

def normalize_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

predictions_norm = [normalize_text(p) for p in predictions]
references_norm = [normalize_text(r) for r in references]


In [9]:
# Sel 8 — Hitung WER (raw & normalized) pakai jiwer
import jiwer

wer_raw = jiwer.wer(references, predictions)
wer_normalized = jiwer.wer(references_norm, predictions_norm)

print("="*50)
print(f"Jumlah sampel dievaluasi : {len(predictions)}")
print(f"WER (raw)                : {wer_raw*100:.2f}%")
print(f"WER (normalized)         : {wer_normalized*100:.2f}%")
print("="*50)
print("Target pembanding dari eval Colab sebelumnya: WER normalized ~12,74% (dilaporkan sebagai 12,79%)")
print("Kalau angka di atas mendekati (misal selisih < 1-2 poin persen), checkpoint-2000 lokal TERVERIFIKASI cocok.")
print("Kalau selisihnya besar (>5 poin persen), lihat catatan troubleshooting di sel Markdown paling bawah.")


Jumlah sampel dievaluasi : 20
WER (raw)                : 40.60%
WER (normalized)         : 12.03%
Target pembanding dari eval Colab sebelumnya: WER normalized ~12,74% (dilaporkan sebagai 12,79%)
Kalau angka di atas mendekati (misal selisih < 1-2 poin persen), checkpoint-2000 lokal TERVERIFIKASI cocok.
Kalau selisihnya besar (>5 poin persen), lihat catatan troubleshooting di sel Markdown paling bawah.


In [10]:
# Sel 9 (opsional) — Lihat beberapa contoh prediksi vs referensi untuk spot-check manual
for i in range(min(5, len(predictions))):
    print(f"[{i}] REF : {references[i]}")
    print(f"[{i}] PRED: {predictions[i]}")
    print("-"*40)


[0] REF : Kamu di mana?
[0] PRED: kamu di mana
----------------------------------------
[1] REF : Tom adalah orang yang menanam mawar di kebun kita.
[1] PRED: tom adalah orang yang menanam mawar di kebun cita
----------------------------------------
[2] REF : Hidup cuma sekali
[2] PRED: hidup cuma sekali
----------------------------------------
[3] REF : Dari mana kamu mendapatkan sepatu aneh itu?
[3] PRED: dari mana kamu mendapatkan sepatu aneh itu
----------------------------------------
[4] REF : Nama "Kleopatra" menjadi persamaan kata dengan wanita yang cantik
[4] PRED: nama cleopatra menjadi persamaan kata dengan wanita yang cantik
----------------------------------------


## Troubleshooting kalau angka WER tidak cocok dengan 12,79%

- **Selisih besar sekali (>20 poin)**: kemungkinan checkpoint yang ke-load salah (cek `CHECKPOINT_PATH` benar-benar checkpoint-2000, bukan basis Whisper-small biasa atau checkpoint lain), atau audio tidak ter-resample dengan benar.
- **Lupa resample 32kHz → 16kHz**: akan menghasilkan WER sangat buruk (>80%) karena Whisper salah membaca durasi/frekuensi. Sel 5 sudah menangani ini otomatis via `librosa.load(..., sr=16000)`.
- **Selisih sedang (5-15 poin)**: kemungkinan besar karena **metode normalisasi teks berbeda** dari yang dipakai di eval Colab dulu (misal dulu juga menghapus angka, atau memakai normalizer bawaan Whisper `BasicTextNormalizer`/`EnglishTextNormalizer`). Coba bandingkan dengan normalizer resmi:
  ```python
  from transformers.models.whisper.english_normalizer import BasicTextNormalizer
  normalizer = BasicTextNormalizer()
  ```
- **Selisih kecil (1-3 poin)**: wajar, bisa karena perbedaan random seed subset sampling atau minor perbedaan versi library — checkpoint dianggap **terverifikasi**.
- **Test set beda jumlah sampel dari 3.694**: pastikan `TEST_CSV_PATH` menunjuk ke split test yang sama persis dengan yang dipakai saat fine-tuning, bukan subset lain.
